In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

The function `range` takes two arguments:
- `start` is an integer,
- `stop` is an integer.

It returns the set of all integers from `start` up to and including `stop`.

In [ ]:
let range (start : int) (stop : int) : int RSet.t =
  RSet.of_list (List.init (stop - start + 1) (fun i -> start + i))

In [ ]:
range 1 5

# The [Zebra Puzzle](https://en.wikipedia.org/wiki/Zebra_Puzzle)

The following puzzle appeared in the magazine *Life International* on the 17th of December in the year 1962:
* There are five houses.
* The Englishman lives in the red house.
* The Spaniard owns the dog.
* Coffee is drunk in the green house.
* The Ukrainian drinks tea.
* The green house is immediately to the right of the ivory house.
* The Old Gold smoker owns snails.
* Kools are smoked in the yellow house.
* Milk is drunk in the middle house.
* The Norwegian lives in the first house.
* The man who smokes Chesterfields lives in the house next to the man with the fox.
* Kools are smoked in the house next to the house where the horse is kept.
* The Lucky Strike smoker drinks orange juice.
* The Japanese smokes Parliaments.
* The Norwegian lives next to the blue house.

Furthermore, each of the five houses is painted in a different colour, their inhabitants are of different nationalities, own different pets, drink different beverages, and smoke different brands of cigarettes.

Our task is to answers the following questions:
1. *Who drinks water?*
2. *Who owns the zebra?*

## Choosing the Appropriate Variables

In order to solve this problem we use the following propositional variables, where the counter $i$ ranges from $1$ to $5$.

* $\texttt{English}\langle i \rangle$ expresses the fact
  that the Englishman lives in house number $i$.

  The remaining nationalities are coded using the variables
  $\texttt{Spanish}\langle i \rangle$, $\texttt{Ukrainian}\langle i \rangle$,
  $\texttt{Norwegian}\langle i \rangle$, and $\texttt{Japanese}\langle i \rangle$.
* $\mathtt{Red}\langle i \rangle$ expresses that house number $i$ is red.

  We use the variables $\texttt{Green}\langle i \rangle$, $\texttt{Ivory}\langle i \rangle$,
  $\texttt{Yellow}\langle i \rangle$, and $\texttt{Blue}\langle i \rangle$ to encode the remaining
  colors.
* $\texttt{OldGold}\langle i \rangle$ expresses that the inhabitant of house number $i$ smokes cigarettes of the brand "Old Gold".

  We use the variables $\texttt{Kools}\langle i \rangle$,
  $\texttt{Chesterfields}\langle i \rangle$, $\texttt{Luckies}\langle i \rangle$, and
  $\texttt{Parliaments}\langle i \rangle$ to encode the cigarette brands.
* $\texttt{Dog}\langle i \rangle$ expresses the fact that the inhabitant of house number
  $i$ keeps a dog as his pet.

  We use the variables $\texttt{Snails}\langle i \rangle$, $\texttt{Fox}\langle i \rangle$,
  $\texttt{Horse}\langle i \rangle$, and $\texttt{Zebra}\langle i \rangle$ to encode the
  remaining pets.
* $\texttt{Coffee}\langle i \rangle$ expresses the fact the the inhabitant of house number $i$
  drinks coffee.

  We use the variables $\texttt{Milk}\langle i \rangle$,
  $\texttt{OrangeJuice}\langle i \rangle$, $\texttt{Tea}\langle i \rangle$,
  and $\texttt{Water}\langle i \rangle$ to encode the remaining drinks.

We are using the angular brackets "$\langle$" and "$\rangle$" as part of the variable names
because our parser for propositional logic accepts these symbols as part of variable names.  The parser would be confused if we would use parentheses.

## Importing the Necessary Notebooks

Our goal is to solve this puzzle by first coding it as a solvability problem of propositional logic and then to solve the resulting set of clauses using the algorithm of Davis and Putnam.  The notebook `06-Davis-Putnam.ipynb` provides the functions `solve` and `complement`.

In [ ]:
import_notebook "06-Davis-Putnam.ipynb"

In order to be able to transform formulas from propositional logic into sets of clauses we need the notebook `04-CNF.ipynb`, which implements the function `normalize` that takes a formula and transforms it into a set of clauses.  This notebook also defines the types `literal`, `clause`, and `cnf`.  As it has already been imported by `06-Davis-Putnam.ipynb`, the following import does nothing.

In [ ]:
import_notebook "04-CNF.ipynb"

In order to write formulas conveniently, we use the parser for propositional logic, which is imported by `04-CNF.ipynb`.

Using the parser and the function `normalize` we can implement a function `parse_knf`.  It takes one argument:
- `s` is a string representing a formula.

It returns a set of clauses that is equivalent to the formula represented by `s`.

In [ ]:
let parse_knf (s : string) : cnf =
  normalize (parse s)

In [ ]:
parse_knf "a → b ↔ b → a"

## Auxiliary Functions

In order to succinctly express the constraints that all houses have different colours, the inhabitants have different nationalities etc., it is convenient to implement a function `at_most_one`.  It takes one argument:
- `s` is a set of propositional variables.

It returns a set of clauses that is true if and only if at most one of the variables from `s` is true:
$$ \texttt{at\_most\_one}(S) :=
   \bigl\{ \{\neg p, \neg q\} \bigm| \langle p, q \rangle \in S \times S \wedge
                                   p \not= q \bigr\}
$$

In [ ]:
let at_most_one (s : string RSet.t) : cnf =
  RSet.flat_map
    (fun p -> RSet.filter_map (fun q -> if p <> q then Some (RSet.of_list [Neg p; Neg q]) else None) s)
    s

The function `var_name` takes two arguments:
- `f` is the name of a property,
- `i` is the number of a house, i.e. $i \in \{1,2,3,4,5\}$.

It returns the string $f\langle i \rangle$, e.g. the call `var_name "Japanese" 2` returns the following string:
```
Japanese<2>
```

In [ ]:
let var_name (f : string) (i : int) : string =
  Printf.sprintf "%s<%d>" f i

In [ ]:
var_name "Japanese" 2

The function `somewhere` takes one argument:
- `x` is the name of a property.

It returns a clause that specifies that the person with property $x$ has to live in one of the houses from $1$ to $5$.  In order to be able to insert this clause into a set of clauses, the result has to be of type `clause`, i.e. a set of literals.

In [ ]:
let somewhere (x : string) : clause =
  RSet.map (fun i -> Pos (var_name x i)) (range 1 5)

In [ ]:
somewhere "Japanese"

The function `at_most_one_at` takes two arguments:
- `s` is an exclusive set of properties,
- `i` is the number of a house.

It returns a set of clauses that specifies that the person living in house number $i$ has at most one of the properties from the set $s$.  For example, if
$s = \{\texttt{"Japanese"}, \texttt{"English"}, \texttt{"Spanish"}, \texttt{"Norwegian"}, \texttt{"Ukrainian"}\}$,
then `at_most_one_at s 3` specifies that the inhabitant of house number 3 has at most one of the nationalities from the set $s$.

In [ ]:
let at_most_one_at (s : string RSet.t) (i : int) : cnf =
  at_most_one (RSet.map (fun x -> var_name x i) s)

In [ ]:
at_most_one_at (RSet.of_list ["A"; "B"; "C"]) 1

The function `one_per_house` takes one argument:
- `s` is an exclusive set of properties, e.g. the set of nationalities

$$\{\texttt{"Japanese"},
       \texttt{"English"},
       \texttt{"Spanish"}, \texttt{"Norwegian"},
       \texttt{"Ukrainian"}\}.
$$
For this set, the function creates a set of clauses that expresses that there has to be a house where the Japanese lives, a house where the Englishman lives, a house where the Spaniard lives, a house where the Norwegian lives, and a house
where the Ukrainian lives.  Furthermore, the set of clauses contains clauses that express that these five persons live in **different** houses.

In [ ]:
let one_per_house (s : string RSet.t) : cnf =
  RSet.union (RSet.map somewhere s) (RSet.flat_map (fun i -> at_most_one_at s i) (range 1 5))

In [ ]:
one_per_house (RSet.of_list ["A"; "B"])

The function `same_house` takes two arguments:
- `a` is the name of a property,
- `b` is the name of a property.

It returns a set of clauses that specifies that if the inhabitant of house number $i$ has the property $a$, then he also has the property $b$ and vice versa.  For example, `same_house "Japanese" "Dog"` specifies that the Japanese guy keeps a dog.

In [ ]:
let same_house (a : string) (b : string) : cnf =
  RSet.flat_map
    (fun i -> parse_knf (Printf.sprintf "%s ↔ %s" (var_name a i) (var_name b i)))
    (range 1 5)

In [ ]:
same_house "Red" "Tea"

The function `next_to` takes two arguments:
- `a` is the name of a property,
- `b` is the name of a property.

It returns a set of clauses that specifies that the inhabitants with properties $a$ and $b$ are direct neighbours.  For example, `next_to "Japanese" "Dog"` specifies that the Japanese guy lives next to the guy who keeps a dog.

**Implementation:** The formulas are collected in a list of strings.  The operator `@` concatenates two lists.  `RSet.unions` combines the sets of clauses of all formulas.

In [ ]:
let next_to (a : string) (b : string) : cnf =
  let formulas =
    [Printf.sprintf "%s → %s" (var_name a 1) (var_name b 2)]
    @ List.map (fun i -> Printf.sprintf "%s → %s ∨ %s" (var_name a i) (var_name b (i - 1)) (var_name b (i + 1))) [2; 3; 4]
    @ [Printf.sprintf "%s → %s" (var_name a 5) (var_name b 4)]
  in
  RSet.unions (List.map parse_knf formulas)

In [ ]:
[1; 2; 3] @ [5; 7] @ [8]

In [ ]:
next_to "A" "B"

The function `left_to` takes two arguments:
- `a` is the name of a property,
- `b` is the name of a property.

It returns a set of clauses that specifies that the inhabitant with property $a$ lives in the house to the left of the inhabitant who has property $b$.  For example, `left_to "Japanese" "Dog"` specifies that the Japanese guy lives in the house to the left of the house where there is a dog.

In [ ]:
let left_to (a : string) (b : string) : cnf =
  let formulas =
    List.map (fun i -> Printf.sprintf "%s → %s" (var_name a i) (var_name b (i + 1))) [1; 2; 3; 4]
    @ ["¬" ^ var_name a 5]
  in
  RSet.unions (List.map parse_knf formulas)

In [ ]:
left_to "A" "B"

In [ ]:
let nations = RSet.of_list ["English"; "Spanish"; "Ukrainian"; "Norwegian"; "Japanese"]
let drinks  = RSet.of_list ["Coffee"; "Tea"; "Milk"; "OrangeJuice"; "Water"]
let pets    = RSet.of_list ["Dog"; "Snails"; "Horse"; "Fox"; "Zebra"]
let brands  = RSet.of_list ["Luckies"; "Parliaments"; "Kools"; "Chesterfields"; "OldGold"]
let colours = RSet.of_list ["Red"; "Green"; "Ivory"; "Yellow"; "Blue"]

The function `all_clauses` takes the argument `()` of type `unit`.  It returns a set of clauses describing the problem.

In [ ]:
let all_clauses () : cnf =
  (* 1. Base constraints: Every property category gets exactly one house *)
  let base = List.map one_per_house [nations; drinks; pets; brands; colours] in
  (* 2. The specific puzzle clues *)
  let puzzle_clues = [
    same_house "English" "Red";          (* The Englishman lives in the red house. *)
    same_house "Spanish" "Dog";          (* The Spaniard owns the dog. *)
    same_house "Coffee" "Green";         (* Coffee is drunk in the green house. *)
    same_house "Ukrainian" "Tea";        (* The Ukrainian drinks tea. *)
    left_to "Ivory" "Green";             (* The green house is immediately to
                                            the right of the ivory house. *)
    same_house "OldGold" "Snails";       (* The Old Gold smoker owns snails. *)
    same_house "Kools" "Yellow";         (* Kools are smoked in the yellow house. *)
    parse_knf (var_name "Milk" 3);       (* Milk is drunk in the middle house. *)
    parse_knf (var_name "Norwegian" 1);  (* The Norwegian lives in the first house. *)
    next_to "Chesterfields" "Fox";       (* The Chesterfields smoker lives next to
                                            the fox owner. *)
    next_to "Kools" "Horse";             (* Kools are smoked next to the horse. *)
    same_house "Luckies" "OrangeJuice";  (* The Lucky Strike smoker drinks orange juice. *)
    same_house "Japanese" "Parliaments"; (* The Japanese smokes Parliaments. *)
    next_to "Norwegian" "Blue"           (* The Norwegian lives next to the blue house. *)
  ] in
  RSet.unions (base @ puzzle_clues)

The function `clause_to_string` takes one argument:
- `c` is a clause.

It converts `c` into a string.  We use it to print all clauses.

In [ ]:
let clause_to_string (c : clause) : string =
  RSet.to_string (function Pos p -> p | Neg p -> "¬" ^ p) c

In [ ]:
let clauses = all_clauses ();;
RSet.iter (fun c -> print_endline (clause_to_string c)) clauses

In [ ]:
RSet.cardinal clauses

The function `main` takes the argument `()`.  It solves the puzzle and returns the solution as a set of unit clauses.

In [ ]:
let main () : cnf =
  solve (all_clauses ())

Solving the problem takes less than a second on my computer.

In [ ]:
let solution = time "Solution" main

## Functions to Pretty Print the Solution

The function `extract_assignment` takes one argument:
- `solution` is a set of unit clauses representing a solution of the puzzle.

It returns an association list that maps every property to the number of the house that has this property.

**Implementation:** The function `arb` from the notebook `06-Davis-Putnam.ipynb` returns an arbitrary element of a set.  As a unit clause has only one element, `arb unit_clause` is the literal of the unit clause.  For every unit clause containing a positive literal, the variable has the form $\textrm{name}\langle i \rangle$ where $i$ is a single digit.  Hence, the house number is the second to last character and the name consists of all characters except the last three.  `RSet.fold` collects the pairs `(name, number)` in a list.

In [ ]:
let extract_assignment (solution : cnf) : (string * int) list =
  RSet.fold
    (fun unit_clause assignment ->
       match arb unit_clause with
       | Pos var ->
           let n      = String.length var in
           let number = int_of_string (String.sub var (n - 2) 1) in
           let name   = String.sub var 0 (n - 3) in
           (name, number) :: assignment
       | Neg _ -> assignment)
    solution []

In [ ]:
extract_assignment solution

In order to display HTML in the notebook, we load the library `jupyter.notebook`, which provides the function `Jupyter_notebook.display`.

In [ ]:
#require "jupyter.notebook";;

The function `show_html` takes one argument:
- `solution` is a set of unit clauses representing a solution of the puzzle.

It displays the solution as an HTML table.  For every house, the table shows the nationality, drink, animal, cigarette brand, and colour of this house.

**Implementation:** For every house and every category, `RSet.filter` selects those properties of the category that are assigned to the house.  As the solution is complete, there is exactly one such property.

In [ ]:
let show_html (solution : cnf) : unit =
  let assignment = extract_assignment solution in
  let html = Buffer.create 4096 in
  Buffer.add_string html "<table style=\"border:2px solid blue\">\n<tr>";
  List.iter
    (fun name -> Buffer.add_string html ("<th style=\"color:gold; background-color:blue\">" ^ name ^ "</th>"))
    ["House"; "Nationality"; "Drink"; "Animal"; "Brand"; "Colour"];
  Buffer.add_string html "</tr>\n";
  for house = 1 to 5 do
    Buffer.add_string html (Printf.sprintf "<tr><td style=\"border:1px solid green\">%d</td>" house);
    List.iter
      (fun category ->
         RSet.iter
           (fun x -> Buffer.add_string html ("<td style=\"border:1px solid green\">" ^ x ^ "</td>"))
           (RSet.filter (fun x -> List.assoc_opt x assignment = Some house) category))
      [nations; drinks; pets; brands; colours];
    Buffer.add_string html "</tr>\n"
  done;
  Buffer.add_string html "</table>";
  ignore (Jupyter_notebook.display "text/html" (Buffer.contents html))

In [ ]:
show_html solution

## Checking the Uniqueness of the Solution

The function `negate_solution` takes one argument:
- `unit_clauses` is a set of unit clauses.

It returns a clause that is the logical negation of `unit_clauses`: The conjunction of the literals of the unit clauses is false if and only if at least one of the complements of these literals is true.  The functions `complement` and `arb` are defined in the notebook `06-Davis-Putnam.ipynb`.

In [ ]:
let negate_solution (unit_clauses : cnf) : clause =
  RSet.map (fun u -> complement (arb u)) unit_clauses

In [ ]:
negate_solution (RSet.of_list [RSet.of_list [Pos "a"]; RSet.of_list [Neg "b"]])

The function `check_uniqueness` takes two arguments:
- `solution` is a set of unit clauses representing a solution of the puzzle,
- `clauses` is the set of clauses describing the puzzle.

It checks whether the solution is unique and prints the result.  To this end, the negation of `solution` is added to `clauses`.  If the resulting set of clauses is unsatisfiable, there is no other solution.

In [ ]:
let check_uniqueness (solution : cnf) (clauses : cnf) : unit =
  let alternative = solve (RSet.add (negate_solution solution) clauses) in
  if RSet.mem RSet.empty alternative then
    print_endline "The solution is unique!"
  else
    print_endline "ERROR: The solution is not unique!"

In [ ]:
time "uniqueness" (fun () -> check_uniqueness solution clauses)